# 🟡 Notebook 3 — Pipeline de Enriquecimiento de Features con LLM
> **Feature engineering con lenguaje natural**

En este notebook vamos a:
1. Tomar un dataset de reseñas de productos con texto libre
2. Usar Gemini para generar features semánticas (sentimiento, intención, urgencia)
3. Comparar el rendimiento de un modelo ML **con y sin** features enriquecidas

**Módulo:** IA Generativa — Clase 4: Pipelines ML + GenAI

## 1. Instalación de dependencias

In [ ]:
!pip install langchain langchain-google-genai scikit-learn pandas numpy python-dotenv -q

## 2. Configuración

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0
)

print("✅ Configuración lista")

## 3. Dataset de reseñas de e-commerce

In [ ]:
# Dataset sintético: reseñas de productos con etiqueta de recompra (target)
datos = [
    {"resena": "Excelente producto, superó mis expectativas. Lo recomiendo a todos mis amigos!", "precio": 45, "dias_envio": 2, "recompra": 1},
    {"resena": "Malísimo, se rompió a la semana. Una basura. Pido devolución urgente.", "precio": 30, "dias_envio": 5, "recompra": 0},
    {"resena": "El producto llegó bien, aunque el empaque estaba un poco dañado. Funciona ok.", "precio": 60, "dias_envio": 3, "recompra": 1},
    {"resena": "No entiendo cómo usar esto, las instrucciones son malísimas. Muy frustrado.", "precio": 25, "dias_envio": 1, "recompra": 0},
    {"resena": "Perfecto para regalo. Mi hijo quedó encantado. Definitivamente volvería a comprar.", "precio": 80, "dias_envio": 2, "recompra": 1},
    {"resena": "Tardó 15 días en llegar cuando decía 3. El producto en sí está bien pero el servicio horrible.", "precio": 55, "dias_envio": 15, "recompra": 0},
    {"resena": "Calidad aceptable para el precio. No es lo mejor del mercado pero cumple su función.", "precio": 20, "dias_envio": 4, "recompra": 1},
    {"resena": "Increíble calidad, materiales premium. Vale cada centavo.", "precio": 120, "dias_envio": 2, "recompra": 1},
    {"resena": "Color completamente diferente al de la foto. Me siento estafado.", "precio": 35, "dias_envio": 3, "recompra": 0},
    {"resena": "Buena relación calidad precio. El envío fue rápido y bien embalado.", "precio": 40, "dias_envio": 2, "recompra": 1},
    {"resena": "Segunda vez que compro este producto. Muy confiable y duradero.", "precio": 65, "dias_envio": 3, "recompra": 1},
    {"resena": "Esperaba más. Para ese precio hay mejores opciones en el mercado.", "precio": 90, "dias_envio": 4, "recompra": 0},
    {"resena": "Me llegó roto. El vendedor no responde. Eviten esta tienda.", "precio": 50, "dias_envio": 7, "recompra": 0},
    {"resena": "Funciona perfecto. Simple, efectivo y fácil de usar. 5 estrellas sin dudarlo.", "precio": 30, "dias_envio": 2, "recompra": 1},
    {"resena": "Regular. Ni muy bien ni muy mal. El envío fue lento pero llegó completo.", "precio": 45, "dias_envio": 8, "recompra": 0},
]

df = pd.DataFrame(datos)
print(f"Dataset: {df.shape}")
print(f"Tasa de recompra: {df['recompra'].mean():.1%}")
df.head()

## 4. Modelo Baseline: Solo features numéricas (sin texto)

In [ ]:
features_base = ["precio", "dias_envio"]
X_base = df[features_base]
y = df["recompra"]

X_train, X_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.3, random_state=42
)

modelo_base = GradientBoostingClassifier(n_estimators=50, random_state=42)
modelo_base.fit(X_train, y_train)

acc_base = accuracy_score(y_test, modelo_base.predict(X_test))
cv_base = cross_val_score(modelo_base, X_base, y, cv=3).mean()

print("📊 MODELO BASELINE (solo features numéricas)")
print(f"  Accuracy test:  {acc_base:.2%}")
print(f"  CV Score (3-fold): {cv_base:.2%}")
print(classification_report(y_test, modelo_base.predict(X_test), 
                             target_names=["No recompra", "Recompra"]))

## 5. Enriquecimiento semántico con Gemini

In [ ]:
prompt_features = ChatPromptTemplate.from_template("""
Analiza la siguiente reseña de e-commerce y extrae features semánticas.
Responde ÚNICAMENTE con un JSON válido, sin markdown ni explicaciones.

Reseña: "{resena}"

Extrae:
{{
  "score_sentimiento": número entre -1.0 (muy negativo) y 1.0 (muy positivo),
  "menciona_devolucion": true o false,
  "menciona_calidad": true o false,
  "menciona_envio": true o false,
  "menciona_precio": true o false,
  "intencion_recompra": número entre 0.0 (nunca volvería) y 1.0 (definitivamente volvería),
  "nivel_frustracion": número entre 0.0 y 1.0,
  "longitud_percibida": "corta" o "media" o "larga"
}}
""")

chain_features = prompt_features | llm | StrOutputParser()

def extraer_features_semanticas(resena: str) -> dict:
    raw = chain_features.invoke({"resena": resena})
    raw = raw.strip().replace("```json", "").replace("```", "").strip()
    return json.loads(raw)

# Procesar todas las reseñas
print("Extrayendo features semánticas...")
features_semanticas = []
for i, resena in enumerate(df["resena"]):
    print(f"  [{i+1}/{len(df)}] {resena[:50]}...", end=" ")
    feats = extraer_features_semanticas(resena)
    features_semanticas.append(feats)
    print(f"sentimiento: {feats['score_sentimiento']:+.2f} ✅")

df_feats = pd.DataFrame(features_semanticas)
print("\nFeatures semánticas extraídas:")
df_feats.head()

## 6. Modelo Enriquecido: Numéricas + Features semánticas

In [ ]:
# Encodear longitud_percibida
le = LabelEncoder()
df_feats["longitud_enc"] = le.fit_transform(df_feats["longitud_percibida"])

# Combinar features
df_enriched = pd.concat([
    df[["precio", "dias_envio"]].reset_index(drop=True),
    df_feats.drop("longitud_percibida", axis=1).reset_index(drop=True)
], axis=1)

# Convertir booleanos
bool_cols = ["menciona_devolucion", "menciona_calidad", 
             "menciona_envio", "menciona_precio"]
for col in bool_cols:
    df_enriched[col] = df_enriched[col].astype(int)

print("Features del modelo enriquecido:")
print(list(df_enriched.columns))

X_enrich = df_enriched
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_enrich, y, test_size=0.3, random_state=42
)

modelo_enrich = GradientBoostingClassifier(n_estimators=50, random_state=42)
modelo_enrich.fit(X_train_e, y_train_e)

acc_enrich = accuracy_score(y_test_e, modelo_enrich.predict(X_test_e))
cv_enrich = cross_val_score(modelo_enrich, X_enrich, y, cv=3).mean()

print("\n📊 MODELO ENRIQUECIDO (numéricas + semánticas)")
print(f"  Accuracy test:  {acc_enrich:.2%}")
print(f"  CV Score (3-fold): {cv_enrich:.2%}")
print(classification_report(y_test_e, modelo_enrich.predict(X_test_e),
                             target_names=["No recompra", "Recompra"]))

## 7. Comparación y feature importance

In [ ]:
print("\n" + "=" * 55)
print("📈 COMPARACIÓN DE MODELOS")
print("=" * 55)
print(f"  {'Modelo':<35} {'CV Score':>10}")
print("-" * 55)
print(f"  {'Baseline (precio + dias_envio)':<35} {cv_base:.2%}")
print(f"  {'Enriquecido (+ LLM features)':<35} {cv_enrich:.2%}")
print(f"  {'Mejora':<35} {(cv_enrich - cv_base):+.2%}")
print("=" * 55)

# Feature importance del modelo enriquecido
importances = pd.DataFrame({
    "feature": df_enriched.columns,
    "importance": modelo_enrich.feature_importances_
}).sort_values("importance", ascending=False)

print("\n🔑 TOP FEATURES POR IMPORTANCIA:")
for _, row in importances.iterrows():
    bar = "█" * int(row["importance"] * 40)
    print(f"  {row['feature']:<30} {bar} {row['importance']:.3f}")

## 🎯 Conclusiones

- El LLM transforma texto no estructurado en **features numéricas útiles** para ML
- Features como `score_sentimiento` e `intencion_recompra` capturan señales que datos numéricos no pueden
- **Patrón clave:** `Texto → LLM → Features semánticas → Concatenar con features existentes → Modelo ML`
- Esto es especialmente valioso cuando tienes datos tabulares + columnas de texto